<a href="https://colab.research.google.com/github/pmadhyastha/INM434/blob/main/Lab8_Scaling_and_Alignmentipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
__author__ = "Pranava Madhyastha"
__version__ = "INM434/IN3045 City, University of London, Spring 2026"

This notebook accompanies Lecture 8. Each section maps to a specific part of Jurafsky & Martin (2026), Chapters 7 and 9. Please work through each section, run the code, and answer the questions at the end of each part -- also remember that we are following on from Lab 7, so I will assume you have indeed completed Lab 7.

In [ ]:
# setting it all up
!pip install transformers accelerate torch --quiet

# see what accelerate is on huggingface

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
import warnings
warnings.filterwarnings("ignore")

# Check GPU
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB"
      if torch.cuda.is_available() else "")


We load TWO versions of Qwen2.5-3B:
- **Base model**: Trained only with next-token prediction (pretraining)
- **Instruct model**: Further trained with instruction tuning and alignment

In [ ]:
MODEL_BASE = "Qwen/Qwen2.5-3B"
MODEL_INSTRUCT = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE)
model_base = AutoModelForCausalLM.from_pretrained(
    MODEL_BASE, torch_dtype=torch.float16, device_map="auto"
)


tokenizer_instruct = AutoTokenizer.from_pretrained(MODEL_INSTRUCT)
model_instruct = AutoModelForCausalLM.from_pretrained(
    MODEL_INSTRUCT, torch_dtype=torch.float16, device_map="auto"
)


# Token Prediction and Logits (SLP §7.4)

A language model assigns a probability distribution over the *entire vocabulary*
for every position. Let's look inside this process.

In [ ]:
def show_top_predictions(model, tokenizer, text, top_k=10):

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)

    # Logits for the LAST position (predicting the next token)
    logits = outputs.logits[0, -1, :]  # shape: [vocab_size]
    probs = F.softmax(logits, dim=-1)

    # Get top-k
    top_probs, top_indices = torch.topk(probs, top_k)

    print(f"Context: '{text}'")
    print(f"{'Rank':<6} {'Token':<20} {'Probability':<12} {'Logit':<10}")
    print("-" * 50)
    for i, (prob, idx) in enumerate(zip(top_probs, top_indices)):
        token = tokenizer.decode(idx)
        print(f"{i+1:<6} {repr(token):<20} {prob.item():<12.4f} {logits[idx].item():<10.2f}")

    return logits, probs

print("=" * 60)
print("EXAMPLE 1: Predicting after a clear factual context")
print("=" * 60)
logits1, probs1 = show_top_predictions(
    model_base, tokenizer,
    "The capital of France is"
)

print("\n" + "=" * 60)
print("EXAMPLE 2: Predicting after an ambiguous context")
print("=" * 60)
logits2, probs2 = show_top_predictions(
    model_base, tokenizer,
    "I went to the"
)

print("\n" + "=" * 60)
print("EXAMPLE 3: What knowledge can a model learn from prediction?")
print("(J&M §7.1: 'roses, dahlias, and peonies are all kinds of...')")
print("=" * 60)
show_top_predictions(
    model_base, tokenizer,
    "With roses, dahlias, and peonies, I was surrounded by"
)

# TODO!

**Q**: The logit for "Paris" is much higher than for other cities.
What does this tell us about what the model learned during pretraining?
(Hint: Have a read through SLP3 - Section 7.1.)

**Q**: Compare the probability distributions for Examples 1 and 2.
Which is more "peaked" (concentrated on fewer tokens)? Why does this
make sense linguistically?

**Q**: Try changing the context to "The square root of 144 is".
What does the model predict? What kind of knowledge is this?

# Decoding Strategies

We implement three strategies from the textbook:
1. **Greedy decoding** (SLP 7.4.1): Always pick the most probable token
2. **Random sampling** (SLP 7.4.2): Sample according to probability
3. **Temperature sampling** (SLP 7.4.3): Reshape distribution then sample

In [ ]:
def generate_with_strategy(model, tokenizer, prompt, max_new_tokens=50,
                           strategy="greedy", temperature=1.0, top_k=0, top_p=1.0,
                           seed=42):

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    if seed is not None:
        torch.manual_seed(seed)

    if strategy == "greedy":
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    elif strategy == "sample":
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=temperature,
            top_k=top_k if top_k > 0 else 0,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(output[0][inputs.input_ids.shape[1]:],
                                  skip_special_tokens=True)
    return generated

prompt = "Once upon a time in a small village, there lived"

print("=" * 70)
print("GREEDY DECODING")
print("=" * 70)
for i in range(3):
    text = generate_with_strategy(model_base, tokenizer, prompt,
                                   strategy="greedy", seed=i)
    print(f"Run {i+1}: {text[:200]}")
print("\n>> Notice: greedy decoding is DETERMINISTIC — same output every time.\n")

print("=" * 70)
print("RANDOM SAMPLING (SLP §7.4.2)")
print("=" * 70)
for i in range(3):
    text = generate_with_strategy(model_base, tokenizer, prompt,
                                   strategy="sample", temperature=1.0, seed=i)
    print(f"Run {i+1}: {text[:200]}")
print("\n>> Notice: random sampling produces diverse but sometimes incoherent text.\n")

print("=" * 70)
print("TEMPERATURE SAMPLING (SLP §7.4.3)")
print("=" * 70)
for temp in [0.1, 0.5, 0.7, 1.0, 1.5, 2.0]:
    text = generate_with_strategy(model_base, tokenizer, prompt,
                                   strategy="sample", temperature=temp, seed=42)
    print(f"τ={temp:<4}: {text[:150]}")


Let's reproduce SLP figure 7.11: showing how temperature reshapes the distribution for a slightly different sentence.

In [ ]:
def visualise_temperature(model, tokenizer, text, temperatures=[0.1, 0.5, 1.0, 2.0, 5.0]):
    """Visualise how temperature changes the probability distribution."""
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1, :]

    fig, axes = plt.subplots(1, len(temperatures), figsize=(4*len(temperatures), 4))
    fig.suptitle(f"Next-token distribution after: '{text}'", fontsize=14, y=1.02)

    for ax, tau in zip(axes, temperatures):
        probs = F.softmax(logits / tau, dim=-1)
        top_probs, top_idx = torch.topk(probs, 10)
        tokens = [tokenizer.decode(i) for i in top_idx]

        ax.barh(range(10), top_probs.cpu().numpy(), color='steelblue')
        ax.set_yticks(range(10))
        ax.set_yticklabels([repr(t) for t in tokens], fontsize=8)
        ax.set_title(f"τ = {tau}", fontsize=12)
        ax.set_xlim(0, 1)
        ax.invert_yaxis()

    plt.tight_layout()
    plt.savefig("temperature_comparison.png", dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: temperature_comparison.png")

visualise_temperature(model_base, tokenizer, "The meaning of life is")

# TODO

**Q**: Run greedy decoding 5 times with the same prompt. Do you get
different outputs? Why or why not? (SLP §7.4.1)

**Q**: SLP say "greedy decoding is too boring, and random sampling
is too random." What evidence from your experiments supports this?

**Q**: Look at the temperature plot. At τ=0.1, what fraction of the
probability mass is on the top token? At τ=5.0? Explain in your own
words why lower temperature makes the distribution "more greedy."

**Q**: If you were building a system for (a) machine translation and
(b) creative story writing, which temperature would you use for each? Why?


**Q**: We also saw Beam Search in the class - can you implement beam search -- say with smaller vs larger beam sizes?


# The Alginment Problem, understanding Base versus Instruct models

SLP Chapter 9 opens with examples of base models failing to follow
instructions. Let's reproduce this phenomenon and see how instruction
tuning fixes it. Remember, we saw it in the last week too?

In [ ]:
def compare_models(prompt, max_tokens=150):
    """Compare base and instruct model outputs for the same prompt."""
    print(f"PROMPT: {prompt}")
    print("-" * 60)

    # Base model — just continue the text
    base_out = generate_with_strategy(
        model_base, tokenizer, prompt,
        max_new_tokens=max_tokens, strategy="sample",
        temperature=0.7, seed=42
    )
    print(f"BASE MODEL:\n{base_out[:300]}")
    print()

    # Instruct model — format as chat
    messages = [{"role": "user", "content": prompt}]
    chat_text = tokenizer_instruct.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    instruct_out = generate_with_strategy(
        model_instruct, tokenizer_instruct, chat_text,
        max_new_tokens=max_tokens, strategy="sample",
        temperature=0.7, seed=42
    )
    print(f"INSTRUCT MODEL:\n{instruct_out[:300]}")
    print("=" * 60 + "\n")

print("=" * 60)
print("EXPERIMENT: Reproducing J&M's alignment examples")
print("=" * 60 + "\n")

# SLP's exact examples from Ouyang et al. 2022
compare_models("Explain the moon landing to a six year old in a few sentences.")
compare_models("Translate to French: The small dog")
compare_models("What is the capital of Australia?")
compare_models("Write a haiku about machine learning.")
compare_models("Is it safe to eat raw chicken?")

# TODO

**Q**: For the "moon landing" prompt, does the base model explain
the moon landing or do something else? What does the instruct model do?
Why is this difference so important? (SLP §9.1)

**Q**: The base model was trained on vastly more data than the
instruct model's fine-tuning data. Why can a small amount of
instruction data change the model's behavior so dramatically?

**Q**: Try the prompt "How do I pick a lock?". How do the base and
instruct models differ? What does this tell us about safety alignment?

# Few-shot Prompting and In-Context Learning (SLP 7.3)

We have covered in the lectures - that in-context learning is  "learning that improves model performance
but does not involve gradient-based updates." Let's see this in action.

In [ ]:
def run_sentiment(model, tokenizer, text, prompt_prefix="", use_chat=False):
    """Run sentiment classification and return the predicted label."""
    full_prompt = f"""{prompt_prefix}Classify the sentiment of the following review as Positive or Negative.
Review: "{text}"
Sentiment:"""

    if use_chat:
        messages = [{"role": "user", "content": full_prompt}]
        full_prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

    inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1, :]

    # Get probabilities for "Positive" and "Negative"
    pos_tokens = tokenizer.encode(" Positive", add_special_tokens=False)
    neg_tokens = tokenizer.encode(" Negative", add_special_tokens=False)

    probs = F.softmax(logits, dim=-1)
    pos_prob = probs[pos_tokens[0]].item() if pos_tokens else 0
    neg_prob = probs[neg_tokens[0]].item() if neg_tokens else 0

    total = pos_prob + neg_prob
    if total > 0:
        pos_prob /= total
        neg_prob /= total

    return "Positive" if pos_prob > neg_prob else "Negative", pos_prob, neg_prob

# Test sentences
test_cases = [
    ("The acting was superb and the plot kept me engaged throughout.", "Positive"),
    ("A complete waste of two hours. Avoid at all costs.", "Negative"),
    ("Not bad, but not great either. It had some moments.", "Positive"),
    ("The special effects were amazing but the story was thin.", "Positive"),
    ("I fell asleep halfway through. Terrible movie.", "Negative"),
]

# Zero-shot vs Few-shot demonstrations
demonstrations = """Example: "A masterpiece of cinema." -> Positive
Example: "Dull and predictable." -> Negative
Example: "I loved every minute of it." -> Positive
Example: "Boring and overlong." -> Negative

"""

print("=" * 70)
print("ZERO-SHOT vs FEW-SHOT SENTIMENT CLASSIFICATION")
print("=" * 70)
print(f"\n{'Review':<55} {'True':<10} {'Zero-shot':<12} {'Few-shot':<12}")
print("-" * 90)

zero_correct = 0
few_correct = 0

for text, true_label in test_cases:
    # Zero-shot
    pred_zero, p_pos_z, p_neg_z = run_sentiment(
        model_instruct, tokenizer_instruct, text, "", use_chat=True
    )
    # Few-shot
    pred_few, p_pos_f, p_neg_f = run_sentiment(
        model_instruct, tokenizer_instruct, text, demonstrations, use_chat=True
    )

    z_mark = "✓" if pred_zero == true_label else "✗"
    f_mark = "✓" if pred_few == true_label else "✗"
    zero_correct += (pred_zero == true_label)
    few_correct += (pred_few == true_label)

    print(f"{text[:53]:<55} {true_label:<10} {pred_zero} {z_mark:<5} {pred_few} {f_mark}")

print(f"\nZero-shot accuracy: {zero_correct}/{len(test_cases)}")
print(f"Few-shot accuracy:  {few_correct}/{len(test_cases)}")

"""
### Min et al. (2022) experiment: Do demonstrations need correct labels?

J&M cite Min et al. (2022) showing that demonstrations with *incorrect*
labels can still help. Let's test this.
"""

# Deliberately WRONG labels
wrong_demonstrations = """Example: "A masterpiece of cinema." -> Negative
Example: "Dull and predictable." -> Positive
Example: "I loved every minute of it." ->
"""

print("\n" + "=" * 70)
print("EXPERIMENT: Correct vs WRONG demonstration labels")
print("(Testing the Min et al. 2022 finding cited in J&M §7.3)")
print("=" * 70)
print(f"\n{'Review':<55} {'Correct demos':<15} {'WRONG demos':<15}")
print("-" * 85)

for text, true_label in test_cases:
    pred_correct, _, _ = run_sentiment(
        model_instruct, tokenizer_instruct, text, demonstrations, use_chat=True
    )
    pred_wrong, _, _ = run_sentiment(
        model_instruct, tokenizer_instruct, text, wrong_demonstrations, use_chat=True
    )
    print(f"{text[:53]:<55} {pred_correct:<15} {pred_wrong:<15}")

# TODO

**Q**: Does few-shot prompting improve over zero-shot? By how much?

**Q**: When you use WRONG labels in the demonstrations, does the
model follow the wrong labels or still get the answer right? What does
this tell us about what demonstrations actually teach the model?
(SLP §7.3 discusses this.)

**Q**: SLP say the primary benefit of demonstrations is to
"demonstrate the task and the format of the output rather than
demonstrating the right answers." Does your experiment support this?


# ADVANCED TODO

**Q**: Can you experiment with Chain-of-Thought prompting? (SLP 9.4). Use the same setup and examples as in the book.  